In [ ]:
!pip install kaggle -q

from google.colab import files
files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Dataset'i indir
!kaggle datasets download -d bhushandivekar/video-game-sales-and-industry-data-1980-2024
!kaggle datasets download -d thedevastator/video-game-sales-and-ratings
!kaggle datasets download -d anandshaw2001/video-game-sales

#zip dosyaları aç
import zipfile

zip_files = [
    "video-game-sales-and-industry-data-1980-2024.zip",
    "video-game-sales-and-ratings.zip",
    "video-game-sales.zip"
]

for zip_name in zip_files:
    folder_name = zip_name.replace(".zip", "")
    os.makedirs(folder_name, exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(folder_name)

print("Zip dosyalari acildi.")

#hangi dosyalar geldi
print("Klasor icerigi:")
for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root, file))

#datasetleri oku
import pandas as pd

df1 = pd.read_csv("video-game-sales/vgsales.csv")
df2 = pd.read_csv("video-game-sales-and-ratings/Video_Games.csv")
df3 = pd.read_csv("video-game-sales-and-industry-data-1980-2024/Video Games Sales (1980-2024) - Raw.csv")

print("df1:", df1.shape)
print("df2:", df2.shape)
print("df3:", df3.shape)

df1.head()

#sütun adlarını oku
print(df1.columns)
print(df2.columns)
print(df3.columns)

#dataset1 userscore ve criticscore yok
df1 = df1.rename(columns={
    'Name': 'name',
    'Platform': 'platform',
    'Year': 'year',
    'Genre': 'genre',
    'Publisher': 'publisher',
    'Global_Sales': 'global_sales',
    'JP_Sales':'jp_sales'
})

df1['critic_score'] = None
df1['user_score'] = None

df1 = df1[['name','platform','year','genre','publisher','global_sales','jp_sales','critic_score','user_score']]

#dataset2, tam veri
df2 = df2.rename(columns={
    'Name': 'name',
    'Platform': 'platform',
    'Year_of_Release': 'year',
    'Genre': 'genre',
    'Publisher': 'publisher',
    'Global_Sales': 'global_sales',
    'JP_Sales':'jp_sales',
    'Critic_Score': 'critic_score',
    'User_Score': 'user_score'
})

df2 = df2[['name','platform','year','genre','publisher','global_sales','jp_sales','critic_score','user_score']]

#dataset3 userscore yok
df3 = df3.rename(columns={
    'title': 'name',
    'console': 'platform',
    'release_date': 'year',
    'genre': 'genre',
    'publisher': 'publisher',
    'total_sales': 'global_sales',
    'jp_sales':'jp_sales',
    'critic_score': 'critic_score'
})

df3['user_score'] = None

df3 = df3[['name','platform','year','genre','publisher','global_sales','jp_sales','critic_score','user_score']]





In [ ]:

final_df = pd.concat([df1, df2, df3], ignore_index=True)

In [ ]:

dup_rows = final_df[
    final_df.duplicated(
        subset=['name','platform','year','genre','publisher','global_sales','jp_sales','critic_score','user_score'],
        keep=False
    )
]

dup_rows


In [ ]:
#yılları aynı formata çevir, None'ı NaN'e çevir

In [ ]:
final_df['year'] = pd.to_datetime(final_df['year'], errors='coerce', dayfirst=True).dt.year

In [ ]:
print(final_df[['year']].sample(20))

In [ ]:
import numpy as np

# 2. KARMAŞAYI ÇÖZEN KISIM:
# Sütunlarda string olarak yazılmış 'None', 'null' veya manuel eklediğin None'ları
# gerçek np.nan (Not a Number) formatına çekiyoruz.
bosluk_varyasyonlari = ['None', 'none', 'null', 'NULL', 'nan', 'NaN', ' ', 'N/A']
final_df.replace(bosluk_varyasyonlari, np.nan, inplace=True)
# Veri setinden rastgele 20 örnek çeker
final_df.sample(20)

In [ ]:
#eu_sales ve na_sales sütunlarını da ekle